# 🏥 Healthcare Knowledge Navigator

**An Explainable RAG (Retrieval-Augmented Generation) System for Medical Knowledge**

This notebook implements a complete healthcare RAG pipeline with:
- **ChromaDB** for vector storage and retrieval
- **Sentence-Transformers** (`all-MiniLM-L6-v2`) for embeddings
- **Google Gemini Flash** (free tier) for answer generation
- **SHAP** for explainability — shows which documents contribute to each answer
- **Gradio** for an interactive UI

---

> ⚠️ **Disclaimer**: This system is for educational and clinical decision support purposes only. Always verify information against current guidelines.

## Cell 1: Install Dependencies

In [ ]:
# Cell 1: Install all required packages
# chromadb 0.4.24 for Colab compatibility
# sentence-transformers for embeddings
# google-genai for Gemini Flash API
# shap for explainability
# gradio for the interactive UI

!pip install -q \
    chromadb==0.4.24 \
    google-genai \
    sentence-transformers \
    shap \
    gradio \
    scikit-learn \
    matplotlib \
    numpy

print("\n✅ All packages installed successfully!")

## Cell 2: Imports & Gemini API Key

In [ ]:
# Cell 2: Imports and Gemini API key configuration

import hashlib
import json
import logging
import os
import warnings
from getpass import getpass
from io import BytesIO

import chromadb
import gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import shap
from google import genai
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Suppress noisy warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("healthcare-rag")

# --- Gemini API Key (hidden input) ---
print("🔑 Enter your Google Gemini API key below.")
print("   Get a free key at: https://aistudio.google.com/apikey\n")
GEMINI_API_KEY = getpass("Gemini API Key: ")

# Initialize Gemini client
gemini_client = None
GEMINI_MODEL = "gemini-2.5-flash"

if GEMINI_API_KEY:
    try:
        gemini_client = genai.Client(api_key=GEMINI_API_KEY)
        # Quick validation call
        _test = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents="Say OK",
            config={"max_output_tokens": 5},
        )
        print("\n✅ Gemini client initialized and verified!")
    except Exception as e:
        print(f"\n⚠️ Gemini initialization failed: {e}")
        print("   The system will still work but answers will be retrieval-only.")
        gemini_client = None
else:
    print("\n⚠️ No API key provided. Answers will be retrieval-only (no LLM generation).")

print("\n✅ All imports loaded successfully!")

## Cell 3: Sample Medical Knowledge Base

15 comprehensive medical documents covering clinical guidelines, drug information, treatment protocols, and research summaries.

In [ ]:
# Cell 3: Sample Medical Knowledge Base — 15 documents hardcoded as text strings

MEDICAL_DOCUMENTS = {
    "acute_mi_protocol.txt": """---
title: "Acute Myocardial Infarction (STEMI) Treatment Protocol"
source: "ACC/AHA STEMI Guidelines 2023"
category: "Treatment Protocols"
date: "2023-11-01"
evidence_level: "Level A"
specialty: "Cardiology"
---

# Acute ST-Elevation Myocardial Infarction (STEMI) Protocol

## Initial Assessment (Door-to-ECG ≤10 minutes)
- 12-lead ECG immediately upon arrival
- STEMI criteria: ST elevation ≥1 mm in ≥2 contiguous leads, or new LBBB
- Activate cardiac catheterization lab immediately

## Immediate Management
1. **Oxygen**: Only if SpO2 <90% (routine O2 not recommended)
2. **Aspirin**: 162–325 mg chewed immediately (non-enteric coated)
3. **P2Y12 inhibitor**: 
   - Ticagrelor 180 mg loading dose (preferred), OR
   - Clopidogrel 600 mg loading dose
4. **Anticoagulation**: Unfractionated heparin (UFH) 60 units/kg IV bolus (max 4,000 units), then 12 units/kg/hr (max 1,000 units/hr)
5. **Nitroglycerin**: 0.4 mg SL every 5 min x 3 for ongoing chest pain (avoid if SBP <90, RV infarction, or PDE5 inhibitor use within 24-48 hours)
6. **Morphine**: 2–4 mg IV for pain not relieved by nitroglycerin (use cautiously)
7. **Beta-blocker**: Metoprolol 5 mg IV x 3 (avoid if hypotensive, bradycardic, or signs of HF)

## Primary PCI (Preferred Reperfusion Strategy)
- Door-to-balloon time goal: ≤90 minutes
- Transfer to PCI-capable center if door-to-balloon ≤120 minutes
- Drug-eluting stent preferred over bare-metal stent
- Radial artery access preferred over femoral

## Fibrinolytic Therapy (If PCI not available within 120 minutes)
- Door-to-needle time goal: ≤30 minutes
- Tenecteplase (TNK): Weight-based single IV bolus
  - <60 kg: 30 mg; 60–69 kg: 35 mg; 70–79 kg: 40 mg; 80–89 kg: 45 mg; ≥90 kg: 50 mg
- Contraindications: Active bleeding, prior ICH, ischemic stroke within 3 months, suspected aortic dissection
- Assess for reperfusion at 60–90 minutes; transfer for rescue PCI if unsuccessful

## Post-PCI/Hospital Management
- Dual antiplatelet therapy (DAPT): Aspirin + P2Y12 inhibitor for minimum 12 months
- High-intensity statin: Atorvastatin 80 mg daily
- ACE inhibitor: Within 24 hours (especially if anterior MI, HF, LVEF <40%)
- Beta-blocker: Within 24 hours if hemodynamically stable
- Aldosterone antagonist: If LVEF ≤40% + HF symptoms or diabetes
- Echocardiography: Assess LV function before discharge

## Secondary Prevention
- Cardiac rehabilitation referral
- Smoking cessation counseling
- Blood pressure goal <130/80 mmHg
- LDL goal <70 mg/dL (or ≥50% reduction)
- HbA1c goal <7% for diabetic patients
- Mediterranean or DASH diet""",

    "hypertension_guidelines.txt": """---
title: "Hypertension Management Guidelines (JNC 8 Summary)"
source: "Eighth Joint National Committee (JNC 8)"
category: "Clinical Guidelines"
date: "2024-01-15"
evidence_level: "Level A"
specialty: "Cardiology"
---

# Hypertension Management Guidelines

## Definition and Classification
Hypertension is defined as a sustained systolic blood pressure (SBP) ≥140 mmHg or diastolic blood pressure (DBP) ≥90 mmHg. The classification system includes:
- Normal: SBP <120 mmHg and DBP <80 mmHg
- Elevated: SBP 120-129 mmHg and DBP <80 mmHg
- Stage 1 Hypertension: SBP 130-139 mmHg or DBP 80-89 mmHg
- Stage 2 Hypertension: SBP ≥140 mmHg or DBP ≥90 mmHg

## Treatment Thresholds
- General population aged ≥60 years: Initiate treatment at SBP ≥150 mmHg or DBP ≥90 mmHg. Target: SBP <150 mmHg and DBP <90 mmHg.
- General population aged <60 years: Initiate treatment at SBP ≥140 mmHg or DBP ≥90 mmHg. Target: SBP <140 mmHg and DBP <90 mmHg.
- Patients with diabetes: Initiate treatment at SBP ≥140 mmHg or DBP ≥90 mmHg. Target: SBP <140 mmHg and DBP <90 mmHg.
- Patients with CKD: Initiate treatment at SBP ≥140 mmHg or DBP ≥90 mmHg. Target: SBP <140 mmHg and DBP <90 mmHg.

## First-Line Pharmacotherapy
For the general non-Black population (including those with diabetes):
1. Thiazide-type diuretic (e.g., chlorthalidone 12.5–25 mg daily)
2. Calcium channel blocker (CCB) (e.g., amlodipine 5–10 mg daily)
3. ACE inhibitor (e.g., lisinopril 10–40 mg daily)
4. Angiotensin receptor blocker (ARB) (e.g., losartan 50–100 mg daily)

For the general Black population (including those with diabetes):
1. Thiazide-type diuretic
2. Calcium channel blocker (CCB)

For patients with CKD (regardless of race):
- Initial therapy should include an ACE inhibitor or ARB to improve kidney outcomes.

## Lifestyle Modifications
- DASH diet: Rich in fruits, vegetables, whole grains, and low-fat dairy; reduced saturated fat
- Sodium restriction: <2,400 mg/day (ideally <1,500 mg/day)
- Physical activity: 90–150 minutes of aerobic exercise per week
- Weight management: Maintain BMI 18.5–24.9 kg/m²
- Alcohol limitation: ≤2 drinks/day for men, ≤1 drink/day for women
- Smoking cessation

## Combination Therapy
If blood pressure goal is not reached within one month:
- Increase dose of initial drug, OR
- Add a second drug from the recommended classes
- If goal not reached with two drugs, add a third drug
- Do NOT combine ACE inhibitors with ARBs

## Monitoring
- Recheck blood pressure within 1 month of initiating or changing therapy
- Once at goal, monitor every 3–6 months
- Monitor serum creatinine and potassium 1–2 times per year""",

    "diabetes_management.txt": """---
title: "Type 2 Diabetes Management (ADA Standards of Care)"
source: "American Diabetes Association (ADA) Standards of Care 2024"
category: "Clinical Guidelines"
date: "2024-01-01"
evidence_level: "Level A"
specialty: "Endocrinology"
---

# Type 2 Diabetes Management

## Diagnosis
Type 2 diabetes is diagnosed by any of the following criteria:
- Fasting plasma glucose (FPG) ≥126 mg/dL (7.0 mmol/L)
- 2-hour plasma glucose ≥200 mg/dL (11.1 mmol/L) during oral glucose tolerance test (OGTT)
- Hemoglobin A1c (HbA1c) ≥6.5%
- Random plasma glucose ≥200 mg/dL with classic symptoms of hyperglycemia

## Glycemic Targets
- HbA1c target: <7.0% for most non-pregnant adults
- More stringent target (<6.5%): For patients with short disease duration, long life expectancy, and no significant cardiovascular disease
- Less stringent target (<8.0%): For patients with history of severe hypoglycemia, limited life expectancy, advanced complications

## First-Line Therapy
1. Metformin: First-line pharmacotherapy for type 2 diabetes
   - Starting dose: 500 mg once or twice daily with meals
   - Maximum dose: 2,000–2,550 mg/day
   - Contraindicated in eGFR <30 mL/min/1.73 m²
   - Monitor vitamin B12 levels with long-term use

## Second-Line Therapy (Add to Metformin)
Selection based on patient-specific factors:

For patients with established ASCVD or high risk:
- GLP-1 receptor agonist (liraglutide, semaglutide) with proven cardiovascular benefit
- SGLT2 inhibitor (empagliflozin, dapagliflozin) with proven cardiovascular benefit

For patients with heart failure:
- SGLT2 inhibitor (regardless of HbA1c)

For patients with CKD:
- SGLT2 inhibitor (if eGFR adequate) or GLP-1 receptor agonist

For patients needing weight management:
- GLP-1 receptor agonist (semaglutide, tirzepatide) — significant weight loss benefit
- SGLT2 inhibitor — modest weight loss

For patients where cost is a major issue:
- Sulfonylurea (glipizide, glimepiride) or thiazolidinedione (pioglitazone)

## Insulin Therapy
Indications for insulin:
- Symptomatic hyperglycemia
- HbA1c >10% or blood glucose ≥300 mg/dL
- Type 1 diabetes features (ketosis)

Basal insulin options:
- Insulin glargine (Lantus): Once daily, usually at bedtime
- Insulin detemir (Levemir): Once or twice daily
- Starting dose: 10 units/day or 0.1–0.2 units/kg/day

## Comprehensive Care
- Blood pressure target: <130/80 mmHg
- Lipids: Statin therapy for all patients aged 40–75 with diabetes
- Aspirin: 75–162 mg/day for secondary prevention
- Eye exam: Annual dilated eye exam
- Foot exam: Comprehensive foot evaluation at least annually
- Kidney monitoring: Annual urinary albumin and eGFR
- Vaccinations: Influenza, pneumococcal, hepatitis B, COVID-19""",

    "heart_failure_guidelines.txt": """---
title: "Heart Failure Management (ACC/AHA Guidelines)"
source: "American College of Cardiology/American Heart Association (ACC/AHA) 2022"
category: "Clinical Guidelines"
date: "2024-03-01"
evidence_level: "Level A"
specialty: "Cardiology"
---

# Heart Failure Management Guidelines

## Classification
Heart failure is classified by left ventricular ejection fraction (LVEF):
- HFrEF (Heart Failure with Reduced EF): LVEF ≤40%
- HFmrEF (Heart Failure with Mildly Reduced EF): LVEF 41–49%
- HFpEF (Heart Failure with Preserved EF): LVEF ≥50%

## Stages of Heart Failure
- Stage A: At risk for HF (hypertension, diabetes, CAD, family history)
- Stage B: Pre-HF (structural heart disease, elevated biomarkers, no symptoms)
- Stage C: Symptomatic HF
- Stage D: Advanced HF (refractory symptoms despite optimal therapy)

## NYHA Functional Classification
- Class I: No limitation of physical activity
- Class II: Slight limitation; comfortable at rest, symptoms with ordinary activity
- Class III: Marked limitation; comfortable at rest, symptoms with less-than-ordinary activity
- Class IV: Unable to carry out any physical activity without discomfort; symptoms at rest

## Guideline-Directed Medical Therapy (GDMT) for HFrEF

### Four Pillars of Therapy (initiate simultaneously when possible):
1. **ACE inhibitor/ARB or ARNI**: 
   - Sacubitril/valsartan (Entresto) preferred over ACE inhibitor/ARB
   - Starting dose: Sacubitril/valsartan 24/26 mg twice daily, titrate to 97/103 mg twice daily
   
2. **Beta-blocker** (evidence-based):
   - Carvedilol: Start 3.125 mg twice daily, target 25 mg twice daily
   - Metoprolol succinate: Start 12.5–25 mg daily, target 200 mg daily
   - Bisoprolol: Start 1.25 mg daily, target 10 mg daily

3. **Mineralocorticoid Receptor Antagonist (MRA)**:
   - Spironolactone 25–50 mg daily or Eplerenone 25–50 mg daily
   - Monitor potassium (contraindicated if K+ >5.0 mEq/L)
   - Monitor renal function (avoid if eGFR <30 mL/min)

4. **SGLT2 inhibitor**:
   - Dapagliflozin 10 mg daily or Empagliflozin 10 mg daily
   - Benefit regardless of diabetes status
   - Reduces HF hospitalizations and cardiovascular death

### Additional Therapies:
- Diuretics: For volume overload/congestion (furosemide, bumetanide, torsemide)
- Hydralazine/isosorbide dinitrate: For Black patients already on optimal therapy, or if ACEi/ARB/ARNI not tolerated
- Ivabradine: If heart rate ≥70 bpm on maximum tolerated beta-blocker
- Digoxin: May reduce hospitalizations (target level 0.5–0.9 ng/mL)

## Device Therapy
- ICD: Primary prevention if LVEF ≤35% despite ≥3 months GDMT, NYHA II–III, life expectancy >1 year
- CRT: If LVEF ≤35%, NYHA II–IV, LBBB with QRS ≥150 ms on optimal GDMT

## Management of HFpEF
- SGLT2 inhibitors: Recommended (Class 2a recommendation)
- Diuretics: For volume management
- Manage comorbidities: Hypertension, atrial fibrillation, obesity, CAD
- Exercise training: Improves functional capacity and quality of life

## Monitoring
- Monitor weight daily
- BNP/NT-proBNP trends
- Renal function and electrolytes regularly
- Assess volume status at each visit
- Cardiac rehabilitation referral""",

    "atrial_fibrillation.txt": """---
title: "Atrial Fibrillation Management"
source: "AHA/ACC/HRS Guideline for Management of AF 2023"
category: "Clinical Guidelines"
date: "2024-01-20"
evidence_level: "Level A"
specialty: "Cardiology"
---

# Atrial Fibrillation Management

## Classification
- Paroxysmal AF: Self-terminating, usually within 48 hours (max 7 days)
- Persistent AF: Continuous AF lasting >7 days or requiring cardioversion
- Long-standing persistent AF: Continuous AF >12 months
- Permanent AF: AF accepted by patient and physician; no further rhythm control attempts

## Stroke Risk Assessment — CHA₂DS₂-VASc Score
- C: Congestive heart failure (+1)
- H: Hypertension (+1)
- A₂: Age ≥75 years (+2)
- D: Diabetes mellitus (+1)
- S₂: Prior stroke/TIA/thromboembolism (+2)
- V: Vascular disease (prior MI, PAD, aortic plaque) (+1)
- A: Age 65–74 years (+1)
- Sc: Sex category — female (+1)

### Anticoagulation Recommendations:
- Score 0 (males) or 1 (females): No anticoagulation
- Score 1 (males): Consider anticoagulation
- Score ≥2 (males) or ≥3 (females): Anticoagulation recommended

## Anticoagulation Options

### Direct Oral Anticoagulants (DOACs) — PREFERRED over warfarin:
1. **Apixaban (Eliquis)**: 5 mg BID (reduce to 2.5 mg BID if ≥2 of: age ≥80, weight ≤60 kg, Cr ≥1.5)
2. **Rivaroxaban (Xarelto)**: 20 mg daily with evening meal (15 mg if CrCl 15–50)
3. **Dabigatran (Pradaxa)**: 150 mg BID (75 mg BID if CrCl 15–30)
4. **Edoxaban (Savaysa)**: 60 mg daily (30 mg if CrCl 15–50 or weight ≤60 kg)

### Rate Control
- First-line: Beta-blocker (metoprolol, atenolol) or non-dihydropyridine CCB (diltiazem, verapamil)
- Heart failure (HFrEF): Beta-blocker (avoid non-DHP CCB) ± digoxin
- Lenient rate target: Resting HR <110 bpm (RACE II trial)
- Strict rate target: Resting HR <80 bpm (consider if symptomatic)

### Catheter Ablation:
- Recommended for symptomatic AF refractory to ≥1 antiarrhythmic drug
- First-line therapy: Reasonable for paroxysmal AF (EARLY-AF, STOP AF First trials)
- Pulmonary vein isolation (PVI) is the cornerstone technique
- Success rate: 70–80% single procedure (paroxysmal AF)

## Lifestyle and Risk Factor Management
- Weight loss: ≥10% body weight reduces AF burden (LEGACY trial)
- Exercise: Moderate regular exercise reduces AF recurrence
- Alcohol reduction: Abstinence reduces AF recurrence (ALCOHOL-AF trial)
- Sleep apnea treatment: CPAP reduces AF recurrence""",

    "copd_management.txt": """---
title: "COPD Management Guidelines"
source: "GOLD 2024 Report (Global Initiative for Chronic Obstructive Lung Disease)"
category: "Clinical Guidelines"
date: "2024-01-01"
evidence_level: "Level A"
specialty: "Pulmonology"
---

# COPD Management Guidelines

## Definition
COPD is a heterogeneous lung condition characterized by chronic respiratory symptoms (dyspnea, cough, sputum production) due to airway and/or alveolar abnormalities, usually caused by significant exposure to noxious particles or gases.

## Diagnosis
- Spirometry required: Post-bronchodilator FEV1/FVC <0.7 confirms airflow limitation
- Symptoms: Dyspnea, chronic cough, sputum production
- Risk factors: Smoking (primary), occupational exposures, biomass fuel, alpha-1 antitrypsin deficiency

## GOLD Classification (Severity of Airflow Limitation)
- GOLD 1 (Mild): FEV1 ≥80% predicted
- GOLD 2 (Moderate): 50% ≤ FEV1 < 80% predicted
- GOLD 3 (Severe): 30% ≤ FEV1 < 50% predicted
- GOLD 4 (Very Severe): FEV1 <30% predicted

## Pharmacotherapy

### Group A:
- Bronchodilator (short or long-acting): SABA (albuterol PRN) or SAMA (ipratropium PRN)

### Group B:
- LABA (salmeterol, formoterol, indacaterol, olodaterol) OR
- LAMA (tiotropium, umeclidinium, glycopyrronium)
- If persistent dyspnea: LABA + LAMA combination

### Group E:
- LABA + LAMA combination (preferred initial therapy)
- If blood eosinophils ≥300 cells/μL: Consider LABA + LAMA + ICS (triple therapy)
- Triple therapy options: Fluticasone/umeclidinium/vilanterol (Trelegy) or budesonide/glycopyrrolate/formoterol (Breztri)
- IMPACT trial: Triple therapy reduced exacerbations by 15% and mortality by 28% vs. LAMA/LABA

## Non-Pharmacological Management
- Smoking cessation: MOST important intervention (reduces FEV1 decline)
- Pulmonary rehabilitation: For all patients with significant dyspnea (mMRC ≥2)
- Vaccinations: Influenza, pneumococcal (PCV20), COVID-19, Tdap, RSV (age ≥60)
- Oxygen therapy: If PaO2 ≤55 mmHg or SpO2 ≤88% at rest (LTOT improves survival)

## Acute Exacerbation (AECOPD) Management
- SABA: Albuterol 2.5 mg nebulized every 1–4 hours
- Systemic corticosteroids: Prednisone 40 mg daily for 5 days (REDUCE trial)
- Antibiotics: If purulent sputum or requiring mechanical ventilation
- Oxygen: Target SpO2 88–92% (avoid hyperoxia → CO2 retention)""",

    "asthma_guidelines.txt": """---
title: "Asthma Management Guidelines (GINA 2024)"
source: "Global Initiative for Asthma (GINA) 2024 Report"
category: "Clinical Guidelines"
date: "2024-05-01"
evidence_level: "Level A"
specialty: "Pulmonology"
---

# Asthma Management Guidelines

## Definition
Asthma is a heterogeneous disease characterized by chronic airway inflammation, defined by history of respiratory symptoms (wheeze, shortness of breath, chest tightness, cough) that vary over time and in intensity, together with variable expiratory airflow limitation.

## Diagnosis
- Spirometry showing variable expiratory airflow limitation:
  - FEV1/FVC ratio reduced (<0.75–0.80 in adults)
  - Positive bronchodilator reversibility test: increase in FEV1 >12% and >200 mL from baseline
- Peak expiratory flow (PEF) variability >10%

## Stepwise Treatment Approach

Step 1 (Mild Intermittent):
- Preferred: As-needed low-dose ICS-formoterol

Step 2 (Mild Persistent):
- Preferred: Daily low-dose ICS (e.g., budesonide 200–400 mcg/day) + as-needed SABA

Step 3 (Moderate Persistent):
- Preferred: Low-dose ICS-LABA (e.g., fluticasone/salmeterol 100/50 mcg twice daily)

Step 4 (Severe Persistent):
- Preferred: Medium/high-dose ICS-LABA
- Add-on: Tiotropium, leukotriene modifier, or theophylline

Step 5 (Severe Uncontrolled):
- Preferred: High-dose ICS-LABA + LAMA (tiotropium)
- Add-on: Biologic therapy (omalizumab, mepolizumab, dupilumab, benralizumab)

## Acute Exacerbation Management
- Oxygen: Maintain SpO2 94–98%
- SABA: Salbutamol 4–10 puffs via MDI+spacer, repeated every 20 minutes for 1 hour
- Systemic corticosteroids: Prednisolone 40–50 mg daily for 5–7 days
- Ipratropium bromide: For severe exacerbations
- Magnesium sulfate IV: For life-threatening exacerbations

## Monitoring
- Assess asthma control at every visit using validated tools (ACT, ACQ)
- Review inhaler technique at every visit
- Spirometry at least annually
- Step down treatment after 3 months of good control""",

    "ckd_management.txt": """---
title: "Chronic Kidney Disease Management"
source: "KDIGO Clinical Practice Guidelines 2024"
category: "Clinical Guidelines"
date: "2024-03-15"
evidence_level: "Level A"
specialty: "Nephrology"
---

# Chronic Kidney Disease (CKD) Management

## Definition and Staging
CKD is defined as abnormalities of kidney structure or function present for >3 months.

### GFR Categories:
- G1: Normal or high GFR (≥90 mL/min/1.73 m²)
- G2: Mildly decreased (60–89)
- G3a: Mildly to moderately decreased (45–59)
- G3b: Moderately to severely decreased (30–44)
- G4: Severely decreased (15–29)
- G5: Kidney failure (<15 or dialysis)

## Blood Pressure Management in CKD
- Target: <120 mmHg systolic (SPRINT trial evidence for non-diabetic CKD)
- For diabetic CKD: <130/80 mmHg
- First-line: ACE inhibitor or ARB (especially with albuminuria ≥30 mg/g)

## Pharmacotherapy to Slow Progression

### SGLT2 Inhibitors (NEW CORNERSTONE):
- Dapagliflozin 10 mg or Empagliflozin 10 mg daily
- Indicated: CKD with eGFR ≥20 mL/min regardless of diabetes status
- DAPA-CKD trial: 39% reduction in sustained eGFR decline or ESKD
- EMPA-KIDNEY trial: 28% reduction in progression of kidney disease

### Finerenone (Non-steroidal MRA):
- For type 2 diabetes with CKD and albuminuria
- Reduces kidney and cardiovascular events (FIDELIO-DKD, FIGARO-DKD trials)

## Complication Management

### Anemia:
- Target hemoglobin: 10–11.5 g/dL (avoid >13 g/dL)
- Iron supplementation first: Target ferritin 200–500 ng/mL, TSAT 20–30%
- ESA therapy (epoetin alfa, darbepoetin): After iron optimization

### Mineral and Bone Disorder (CKD-MBD):
- Phosphorus management: Dietary restriction, phosphate binders
- Vitamin D: Supplement if 25(OH)D <30 ng/mL

## Dialysis Planning
- Refer to nephrologist: eGFR <30 mL/min
- Vascular access planning: eGFR <25 mL/min (AV fistula preferred)
- Initiate dialysis: Based on symptoms, not arbitrary GFR threshold""",

    "sepsis_management.txt": """---
title: "Sepsis and Septic Shock Management (Surviving Sepsis Campaign)"
source: "Surviving Sepsis Campaign Guidelines 2021"
category: "Treatment Protocols"
date: "2023-06-01"
evidence_level: "Level A"
specialty: "Critical Care"
---

# Sepsis and Septic Shock Management

## Definitions
- **Sepsis**: Life-threatening organ dysfunction caused by a dysregulated host response to infection. Defined as suspected infection + acute increase in SOFA score ≥2.
- **Septic Shock**: Sepsis with persistent hypotension requiring vasopressors to maintain MAP ≥65 mmHg AND serum lactate >2 mmol/L despite adequate fluid resuscitation.

## Quick SOFA (qSOFA) Screening
Bedside screening for sepsis (≥2 criteria suggests higher mortality risk):
- Respiratory rate ≥22/min
- Altered mentation (GCS <15)
- Systolic blood pressure ≤100 mmHg

## Hour-1 Bundle (Initiate within 1 hour)
1. **Measure lactate level**: Remeasure within 2–4 hours if initial lactate >2 mmol/L
2. **Obtain blood cultures**: Before administering antibiotics (at least 2 sets)
3. **Administer broad-spectrum antibiotics**: 
   - Community-acquired: Piperacillin-tazobactam 4.5g IV OR meropenem 1g IV
   - Hospital-acquired/MRSA risk: Add vancomycin 25–30 mg/kg IV loading dose
4. **Begin rapid fluid resuscitation**: 30 mL/kg crystalloid (Lactated Ringer's preferred)
5. **Apply vasopressors**: If hypotensive during or after fluid resuscitation, target MAP ≥65 mmHg

## Vasopressor Therapy
First-line: **Norepinephrine** 0.1–0.5 mcg/kg/min (preferred)
Second-line: Add **vasopressin** 0.03 units/min
Third-line: Add **epinephrine** 0.1–0.5 mcg/kg/min

## Corticosteroids
- **Hydrocortisone 200 mg/day IV** (50 mg IV q6h or continuous infusion)
- Indicated when adequate fluid resuscitation and vasopressor therapy fail to restore hemodynamic stability

## Monitoring
- Lactate clearance (target ≥10% decrease per 2 hours)
- Urine output ≥0.5 mL/kg/hr
- Mean arterial pressure ≥65 mmHg
- Serial organ function assessment (SOFA score)""",

    "covid19_treatment.txt": """---
title: "COVID-19 Treatment Guidelines Summary"
source: "NIH COVID-19 Treatment Guidelines Panel, WHO Living Guidelines"
category: "Research Summaries"
date: "2024-06-01"
evidence_level: "Level A"
specialty: "Infectious Disease"
---

# COVID-19 Treatment Guidelines

## Risk Stratification
### High-Risk Factors for Severe Disease:
- Age ≥65 years
- Immunocompromised status
- Chronic kidney disease, cardiovascular disease, chronic lung disease, diabetes
- Obesity (BMI ≥30)

## Outpatient Management

### Mild-to-Moderate COVID-19 (High-Risk Patients):
Priority order for antiviral therapy (initiate within 5 days of symptom onset):

1. **Nirmatrelvir/ritonavir (Paxlovid)** — PREFERRED
   - Dose: Nirmatrelvir 300 mg + ritonavir 100 mg orally twice daily for 5 days
   - 89% reduction in hospitalization/death (EPIC-HR trial)

2. **Remdesivir** (3-day outpatient course)
   - Day 1: 200 mg IV, Days 2–3: 100 mg IV daily
   - 87% reduction in hospitalization/death (PINETREE trial)

3. **Molnupiravir** (if others unavailable/contraindicated)
   - 800 mg orally twice daily for 5 days
   - 30% reduction in hospitalization/death (MOVe-OUT trial)

## Hospitalized Patients

### Moderate (Supplemental oxygen needed):
- Remdesivir + Dexamethasone 6 mg daily (IV or PO) for up to 10 days
- RECOVERY trial: Dexamethasone reduced 28-day mortality by 18% overall

### Severe/Critical (High-flow O2, NIV, or mechanical ventilation):
- Dexamethasone 6 mg daily for up to 10 days
- Consider adding: Baricitinib 4 mg PO daily or Tocilizumab 8 mg/kg IV (single dose)

## Adjunctive Therapies
- Anticoagulation: Prophylactic-dose heparin for all hospitalized patients
- Prone positioning: Recommended for ARDS
- Avoid: Hydroxychloroquine, ivermectin, azithromycin (no proven benefit)""",

    "metformin_drug_info.txt": """---
title: "Metformin - Comprehensive Drug Information"
source: "FDA Prescribing Information, ADA Standards"
category: "Drug Information"
date: "2024-02-01"
evidence_level: "Level A"
specialty: "Endocrinology"
---

# Metformin (Glucophage)

## Mechanism of Action
Metformin is a biguanide that lowers blood glucose through multiple mechanisms:
1. Reduces hepatic glucose production (primary mechanism)
2. Increases insulin sensitivity in peripheral tissues (muscle, adipose)
3. Reduces intestinal absorption of glucose
4. Does NOT stimulate insulin secretion (no hypoglycemia as monotherapy)

## Indications
- Type 2 diabetes mellitus (first-line therapy)
- Prediabetes / diabetes prevention (off-label; 31% risk reduction in DPP trial)
- Polycystic ovary syndrome (PCOS) — off-label
- Gestational diabetes — off-label (insulin preferred)

## Dosing
### Immediate Release:
- Starting: 500 mg once or twice daily with meals
- Titration: Increase by 500 mg weekly or 850 mg every 2 weeks
- Maximum: 2,550 mg/day in divided doses (850 mg TID)

### Extended Release:
- Starting: 500 mg once daily with evening meal
- Maximum: 2,000 mg once daily

## Contraindications
- eGFR <30 mL/min/1.73 m² (absolute)
- Acute or chronic metabolic acidosis, including diabetic ketoacidosis

## Adverse Effects
### Common (>10%):
- Gastrointestinal: Nausea, vomiting, diarrhea, flatulence, abdominal discomfort (up to 30%)
- Metallic taste

### Serious:
- Lactic acidosis (rare but potentially fatal; risk 3–10 per 100,000 patient-years)
- Vitamin B12 deficiency (10–30% of long-term users; monitor levels)

## Drug Interactions
- Alcohol: Increases risk of lactic acidosis
- Iodinated contrast media: Hold metformin for 48 hours after contrast
- Cimetidine: Increases metformin levels

## Benefits Beyond Glucose Control
- Weight neutral to slight weight loss (1–3 kg)
- No hypoglycemia risk as monotherapy
- Possible cardiovascular benefit (UKPDS study)
- Low cost; widely available as generic""",

    "lisinopril_drug_info.txt": """---
title: "Lisinopril - Drug Information"
source: "FDA Prescribing Information, Clinical Pharmacology Reference"
category: "Drug Information"
date: "2024-01-15"
evidence_level: "Level A"
specialty: "Cardiology"
---

# Lisinopril

## Classification
- Class: Angiotensin-Converting Enzyme (ACE) Inhibitor
- Brand names: Prinivil, Zestril

## Mechanism of Action
Lisinopril competitively inhibits angiotensin-converting enzyme (ACE), preventing conversion of angiotensin I to angiotensin II. This results in:
1. Decreased aldosterone secretion → reduced sodium and water retention
2. Vasodilation of arterioles and venules → reduced blood pressure
3. Decreased cardiac preload and afterload
4. Reduced cardiac remodeling after myocardial infarction
5. Renoprotective effects via reduced intraglomerular pressure

## Indications
- Hypertension (first-line therapy)
- Heart failure (HFrEF) — reduces mortality and hospitalizations
- Acute myocardial infarction (within 24 hours to improve survival)
- Diabetic nephropathy — slows progression of kidney disease

## Dosing
### Hypertension:
- Starting: 10 mg once daily (5 mg if on diuretic or elderly)
- Usual maintenance: 20–40 mg once daily
- Maximum: 80 mg/day

### Heart Failure:
- Starting: 2.5–5 mg once daily
- Target: 20–40 mg once daily

### Post-MI:
- Starting: 5 mg within 24 hours
- Maintenance: 10 mg once daily

## Contraindications
- History of angioedema related to ACE inhibitor use
- Bilateral renal artery stenosis
- Pregnancy (teratogenic — Category D)

## Adverse Effects
### Common:
- Dry cough (5–20%): Due to bradykinin accumulation
- Dizziness, headache
- Hyperkalemia (especially with renal impairment)

### Serious:
- Angioedema (0.1–0.7%): Higher risk in Black patients
- Acute kidney injury
- Fetal toxicity

## Monitoring
- Blood pressure: 2–4 weeks after initiation or dose change
- Serum creatinine and potassium: At baseline, 1–2 weeks after initiation
- Acceptable creatinine rise: Up to 30% from baseline
- Discontinue if creatinine rises >50% or potassium >5.5 mEq/L""",

    "aspirin_drug_info.txt": """---
title: "Aspirin in Cardiovascular Disease"
source: "AHA/ACC Clinical Practice Guidelines, FDA Label"
category: "Drug Information"
date: "2024-03-01"
evidence_level: "Level A"
specialty: "Cardiology"
---

# Aspirin (Acetylsalicylic Acid) in Cardiovascular Disease

## Mechanism of Action
Aspirin irreversibly inhibits cyclooxygenase-1 (COX-1), blocking thromboxane A2 synthesis in platelets. This results in:
- Inhibition of platelet aggregation for the lifetime of the platelet (7–10 days)
- Antithrombotic effect at low doses (75–325 mg)

## Cardiovascular Indications

### Secondary Prevention (Strong recommendation):
- Acute coronary syndrome (STEMI, NSTEMI, unstable angina)
- Post-PCI/stent placement (part of DAPT)
- Stable ischemic heart disease
- History of ischemic stroke or TIA (non-cardioembolic)
- Peripheral arterial disease

### Primary Prevention:
- NOT routinely recommended for primary prevention (2019 ACC/AHA guideline change)
- May be considered in select patients aged 40–70 with elevated ASCVD risk
- NOT recommended for adults >70 years for primary prevention

## Dosing
- Acute MI/ACS: 162–325 mg chewed immediately, then 81 mg daily
- Secondary prevention: 75–100 mg once daily (81 mg in US)
- Post-PCI with stent: 81 mg daily indefinitely (part of DAPT)

## Dual Antiplatelet Therapy (DAPT)
Aspirin + P2Y12 inhibitor after ACS or PCI:
- Minimum duration: 6–12 months after drug-eluting stent
- Extended DAPT (>12 months): Consider if high ischemic risk

## Adverse Effects
- GI bleeding: Most common serious adverse effect (dose-related)
- Intracranial hemorrhage: Small increased risk
- GI upset, dyspepsia, nausea
- Aspirin-exacerbated respiratory disease

## Drug Interactions
- Warfarin/DOACs: Significantly increased bleeding risk
- NSAIDs (ibuprofen): May antagonize antiplatelet effect
- SSRIs/SNRIs: Increased bleeding risk""",

    "antibiotic_stewardship.txt": """---
title: "Antibiotic Stewardship Principles and Practice"
source: "CDC Core Elements, IDSA Guidelines"
category: "Research Summaries"
date: "2024-02-15"
evidence_level: "Level B"
specialty: "Infectious Disease"
---

# Antibiotic Stewardship Principles

## Core Principles
Antibiotic stewardship programs (ASPs) aim to optimize antimicrobial use to improve patient outcomes, reduce antimicrobial resistance, and decrease healthcare costs.

## The Five Rights of Antibiotic Prescribing
1. **Right drug**: Based on likely pathogen and local resistance patterns
2. **Right dose**: Pharmacokinetic/pharmacodynamic optimization
3. **Right route**: IV-to-oral transition when appropriate
4. **Right duration**: Shortest effective duration
5. **Right time**: Prompt initiation for serious infections

## Common Conditions and Recommended Durations

### Community-Acquired Pneumonia (CAP):
- Minimum 5 days (if clinically stable for ≥48 hours)
- Typical regimen: Amoxicillin 1g TID or Amoxicillin-clavulanate 875/125 mg BID
- Severe/ICU: Ceftriaxone 1g IV daily + azithromycin 500 mg IV daily

### Urinary Tract Infections (UTI):
- Uncomplicated cystitis: Nitrofurantoin 100 mg BID x 5 days (first-line)
- Complicated UTI/Pyelonephritis: Fluoroquinolone x 5–7 days or ceftriaxone 1g IV daily

### Skin and Soft Tissue Infections (SSTI):
- Cellulitis (non-purulent): Cephalexin 500 mg QID x 5–7 days
- Purulent SSTI/abscess: I&D + TMP/SMX DS BID or doxycycline 100 mg BID

## Key Stewardship Interventions
1. **Prospective audit and feedback**: ID review of antibiotic orders
2. **Prior authorization**: Restrict broad-spectrum agents
3. **Antibiogram**: Facility-specific resistance data to guide empiric therapy
4. **De-escalation**: Narrow spectrum based on culture results within 48–72 hours
5. **IV-to-oral conversion**: Switch when clinically improving
6. **Procalcitonin-guided therapy**: Helps differentiate bacterial vs. viral infection

## ESKAPE Pathogens:
- Enterococcus faecium (VRE), Staphylococcus aureus (MRSA)
- Klebsiella pneumoniae (ESBL, CRE), Acinetobacter baumannii (MDR)
- Pseudomonas aeruginosa (MDR), Enterobacter species (AmpC)""",

    "statin_therapy_evidence.txt": """---
title: "Statin Therapy for Cardiovascular Prevention - Evidence Summary"
source: "CTT Collaborators Meta-Analysis, ACC/AHA Guidelines"
category: "Research Summaries"
date: "2024-04-01"
evidence_level: "Level A"
specialty: "Cardiology"
---

# Statin Therapy for Cardiovascular Prevention

## Overview
Statins (HMG-CoA reductase inhibitors) are the cornerstone of lipid-lowering therapy.

## Key Clinical Evidence

### CTT Meta-Analysis (Cholesterol Treatment Trialists)
- Included 26 randomized trials with >170,000 patients
- Each 1 mmol/L (39 mg/dL) reduction in LDL-C reduces:
  - Major vascular events by 22%
  - Major coronary events by 24%
  - Stroke by 15%
  - All-cause mortality by 10%

### JUPITER Trial (Rosuvastatin 20mg)
- 17,802 healthy individuals with LDL <130 mg/dL and hsCRP ≥2 mg/L
- 44% reduction in major cardiovascular events
- NNT: 25 over 5 years for major cardiovascular event prevention

### 4S Trial (Simvastatin 20-40mg)
- 30% reduction in all-cause mortality
- 42% reduction in coronary death

## Statin Intensity Classification

### High-Intensity (lowers LDL ≥50%):
- Atorvastatin 40–80 mg
- Rosuvastatin 20–40 mg

### Moderate-Intensity (lowers LDL 30–49%):
- Atorvastatin 10–20 mg
- Rosuvastatin 5–10 mg
- Simvastatin 20–40 mg

## Indications per ACC/AHA Guidelines
1. Clinical ASCVD (secondary prevention): High-intensity statin
2. LDL ≥190 mg/dL: High-intensity statin
3. Diabetes, age 40–75: Moderate-intensity statin
4. 10-year ASCVD risk ≥7.5%, age 40–75: Moderate-to-high intensity statin

## Adverse Effects
- Myalgia (5–10%): Most common reason for discontinuation
- Rhabdomyolysis: Very rare (<0.1%)
- New-onset diabetes: Small increase; benefits outweigh risk

## Non-Statin Therapies (Add-on if LDL not at goal)
- Ezetimibe: Reduces LDL by additional 18–25%
- PCSK9 inhibitors (evolocumab, alirocumab): Reduce LDL by 50–60%
- Bempedoic acid: Reduces LDL by 15–25%; no myalgia risk
- Inclisiran: siRNA therapy; reduces LDL by 50%; twice-yearly injection""",
}

print(f"✅ Loaded {len(MEDICAL_DOCUMENTS)} medical documents:")
for name in MEDICAL_DOCUMENTS:
    print(f"   📄 {name}")

## Cell 4: Document Ingestion into ChromaDB

In [ ]:
# Cell 4: Document Ingestion into ChromaDB
# Parses metadata, chunks text, embeds with sentence-transformers, stores in ChromaDB

# --- Helper Functions ---

def parse_document_metadata(content):
    """Parse YAML-like frontmatter metadata from document content."""
    metadata = {
        "title": "Unknown",
        "source": "Unknown",
        "category": "General",
        "date": "Unknown",
        "evidence_level": "Not specified",
        "specialty": "General",
    }
    if content.startswith("---"):
        parts = content.split("---", 2)
        if len(parts) >= 3:
            frontmatter = parts[1].strip()
            body = parts[2].strip()
            for line in frontmatter.split("\n"):
                line = line.strip()
                if ":" in line:
                    key, value = line.split(":", 1)
                    key = key.strip().lower().replace(" ", "_")
                    value = value.strip().strip('"').strip("'")
                    if key in metadata:
                        metadata[key] = value
            return metadata, body
    return metadata, content


def chunk_text(text, chunk_size=500, chunk_overlap=100):
    """Split text into overlapping chunks."""
    if len(text) <= chunk_size:
        return [text.strip()] if text.strip() else []

    separators = ["\n\n", "\n", ". ", " "]
    for separator in separators:
        segments = text.split(separator)
        chunks = []
        current_chunk = ""
        for segment in segments:
            test_chunk = current_chunk + separator + segment if current_chunk else segment
            if len(test_chunk) <= chunk_size:
                current_chunk = test_chunk
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = segment
        if current_chunk:
            chunks.append(current_chunk.strip())

        if all(len(c) <= chunk_size for c in chunks):
            # Apply overlap
            final_chunks = []
            for i, chunk in enumerate(chunks):
                if i > 0 and chunk_overlap > 0:
                    prev_text = chunks[i - 1]
                    overlap_text = prev_text[-chunk_overlap:] if len(prev_text) > chunk_overlap else prev_text
                    chunk = overlap_text + " " + chunk
                final_chunks.append(chunk.strip())
            return [c for c in final_chunks if c]

    # Fallback: hard split
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - chunk_overlap
    return chunks


# --- Initialize Models & Database ---

print("⏳ Loading embedding model (all-MiniLM-L6-v2)...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model loaded!")

# ChromaDB 0.3.29 uses the older Client API
print("⏳ Initializing ChromaDB...")
chroma_client = chromadb.Client()

COLLECTION_NAME = "medical_documents"

# Delete existing collection if present, then create fresh
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)
print("✅ ChromaDB collection created!")


# --- Ingest All Documents ---

print("\n⏳ Ingesting documents...")
total_chunks = 0
doc_metadata_map = {}  # filename → metadata

for filename, content in MEDICAL_DOCUMENTS.items():
    metadata, body = parse_document_metadata(content)
    doc_metadata_map[filename] = metadata
    chunks = chunk_text(body)

    if not chunks:
        print(f"  ⚠️ No chunks from {filename}")
        continue

    ids = []
    documents = []
    metadatas = []

    for i, chunk in enumerate(chunks):
        chunk_id = hashlib.md5(f"{filename}::chunk_{i}".encode()).hexdigest()
        ids.append(chunk_id)
        documents.append(chunk)
        metadatas.append({
            "filename": filename,
            "chunk_index": i,
            "total_chunks": len(chunks),
            **metadata,
        })

    # Generate embeddings
    embeddings = embedding_model.encode(documents).tolist()

    # Add to ChromaDB
    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
    )

    total_chunks += len(chunks)
    print(f"  📄 {filename}: {metadata['title']} → {len(chunks)} chunks")

print(f"\n✅ Ingestion complete! {len(MEDICAL_DOCUMENTS)} documents → {total_chunks} chunks indexed.")
print(f"   Collection count: {collection.count()}")

## Cell 5: RAG Pipeline Function

In [ ]:
# Cell 5: RAG Pipeline — Retrieve, Generate, Cite, Score

# --- Prompts ---

SYSTEM_PROMPT = """You are a clinical knowledge assistant for healthcare professionals. Your role is to provide evidence-based medical information by synthesizing the retrieved clinical documents provided to you.

CRITICAL RULES:
1. ONLY use information from the provided context documents. Do NOT use your general knowledge.
2. Always cite your sources using [Source N] notation where N matches the document number in the context.
3. If the context does not contain sufficient information to answer the question, clearly state this limitation.
4. Present information in a structured, clinical format appropriate for healthcare professionals.
5. Include specific dosages, evidence levels, and guideline recommendations when available.
6. Never provide a definitive diagnosis or treatment plan — frame responses as "per guidelines" or "evidence suggests."
7. Highlight any conflicting recommendations between sources.

RESPONSE FORMAT:
- Start with a concise summary answering the question
- Follow with detailed evidence from the provided sources
- Use bullet points for lists of medications, criteria, or steps
- End with any important caveats or limitations of the evidence
- Always include [Source N] citations inline with the relevant information"""

QUERY_TEMPLATE = """Based on the following clinical documents, answer the medical question below.

RETRIEVED CLINICAL DOCUMENTS:
{context}

QUESTION: {query}

Provide a comprehensive, evidence-based answer with inline citations [Source N]. Structure your response with clear sections and bullet points where appropriate."""

DISCLAIMER = (
    "⚠️ DISCLAIMER: This information is provided for educational and clinical decision "
    "support purposes only. It does not constitute medical advice, diagnosis, or treatment "
    "recommendations. Always verify information against current guidelines and consider "
    "individual patient factors."
)


# --- Retrieval ---

def retrieve_chunks(query, top_k=5):
    """Retrieve top-k relevant document chunks from ChromaDB."""
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    chunks = []
    if results and results["ids"] and results["ids"][0]:
        for i, doc_id in enumerate(results["ids"][0]):
            distance = results["distances"][0][i]
            relevance_score = max(0.0, 1.0 - distance / 2.0)
            chunks.append({
                "chunk_id": doc_id,
                "content": results["documents"][0][i],
                "metadata": results["metadatas"][0][i],
                "relevance_score": relevance_score,
            })
    return chunks


# --- Generation ---

def build_context(chunks):
    """Build context string from retrieved chunks for the LLM."""
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        meta = chunk["metadata"]
        context_parts.append(
            f"[Source {i}] — {meta.get('title', 'Unknown')} "
            f"(Source: {meta.get('source', 'Unknown')}, "
            f"Evidence Level: {meta.get('evidence_level', 'N/A')}, "
            f"Category: {meta.get('category', 'General')})\n"
            f"{chunk['content']}\n"
        )
    return "\n---\n".join(context_parts)


def generate_answer(query, chunks):
    """Generate an answer using Gemini given retrieved context."""
    if not gemini_client:
        # Fallback: retrieval-only response
        lines = ["⚠️ *Gemini API not configured. Showing retrieved sources only.*\n"]
        for i, c in enumerate(chunks, 1):
            meta = c["metadata"]
            lines.append(
                f"**[Source {i}]** {meta.get('title', 'Unknown')} "
                f"({meta.get('source', 'Unknown')}): "
                f"relevance {c['relevance_score']:.0%}"
            )
            lines.append(f"> {c['content'][:300]}...\n")
        return "\n".join(lines)

    context = build_context(chunks)
    prompt = QUERY_TEMPLATE.format(context=context, query=query)
    try:
        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config={
                "system_instruction": SYSTEM_PROMPT,
                "temperature": 0.3,
                "max_output_tokens": 2048,
            },
        )
        return response.text
    except Exception as e:
        logger.error(f"Gemini generation error: {e}")
        return f"Error generating response: {str(e)}"


def generate_answer_with_subset(query, all_chunks, mask):
    """Generate answer using a subset of chunks (used by SHAP)."""
    subset = [c for c, m in zip(all_chunks, mask) if m]
    if not subset:
        return ""
    return generate_answer(query, subset)


# --- Confidence Scoring ---

def assess_confidence(chunks):
    """Compute heuristic confidence breakdown from retrieved chunks."""
    if not chunks:
        return {"overall": 0.0, "retrieval_relevance": 0.0,
                "source_agreement": 0.0, "evidence_quality": 0.0, "coverage": 0.0}

    # Retrieval relevance
    relevance_scores = [c["relevance_score"] for c in chunks]
    retrieval_relevance = sum(relevance_scores) / len(relevance_scores)

    # Source agreement (low variance = high agreement)
    if len(relevance_scores) > 1:
        mean_s = sum(relevance_scores) / len(relevance_scores)
        variance = sum((s - mean_s) ** 2 for s in relevance_scores) / len(relevance_scores)
        source_agreement = max(0.0, 1.0 - variance * 4)
    else:
        source_agreement = 0.7

    # Evidence quality
    evidence_map = {"Level A": 1.0, "Level B": 0.8, "Level C": 0.6, "Not specified": 0.4}
    evidence_scores = [
        evidence_map.get(c["metadata"].get("evidence_level", "Not specified"), 0.5)
        for c in chunks
    ]
    evidence_quality = sum(evidence_scores) / len(evidence_scores)

    # Coverage
    unique_titles = set(c["metadata"].get("title", "") for c in chunks)
    coverage = min(1.0, len(unique_titles) / 3.0)

    # Overall weighted score
    overall = (
        retrieval_relevance * 0.35
        + source_agreement * 0.20
        + evidence_quality * 0.25
        + coverage * 0.20
    )

    return {
        "overall": round(min(1.0, max(0.0, overall)), 3),
        "retrieval_relevance": round(min(1.0, max(0.0, retrieval_relevance)), 3),
        "source_agreement": round(min(1.0, max(0.0, source_agreement)), 3),
        "evidence_quality": round(min(1.0, max(0.0, evidence_quality)), 3),
        "coverage": round(min(1.0, max(0.0, coverage)), 3),
    }


# --- Citations ---

def format_citations(chunks):
    """Format citations as a readable markdown string."""
    lines = []
    for i, chunk in enumerate(chunks, 1):
        meta = chunk["metadata"]
        relevance_pct = chunk["relevance_score"] * 100
        lines.append(
            f"**[Source {i}]** {meta.get('title', 'Unknown')}\n"
            f"  - *Source:* {meta.get('source', 'Unknown')}\n"
            f"  - *Category:* {meta.get('category', 'General')} | "
            f"*Evidence:* {meta.get('evidence_level', 'N/A')} | "
            f"*Relevance:* {relevance_pct:.1f}%\n"
        )
    return "\n".join(lines)


# --- Full RAG Pipeline ---

def rag_query(query, top_k=5):
    """
    Full RAG pipeline: Retrieve → Generate → Cite → Score.
    Returns (answer, citations_text, confidence_dict, chunks).
    """
    if not query or len(query.strip()) < 3:
        return "Please enter a medical question (at least 3 characters).", "", {}, []

    # Retrieve
    chunks = retrieve_chunks(query, top_k=top_k)

    if not chunks:
        return (
            "No relevant documents found. Please try rephrasing your question.",
            "",
            {"overall": 0.0},
            [],
        )

    # Generate
    answer = generate_answer(query, chunks)

    # Citations
    citations_text = format_citations(chunks)

    # Confidence
    confidence = assess_confidence(chunks)

    # Add disclaimer
    full_answer = f"{answer}\n\n---\n\n{DISCLAIMER}"

    return full_answer, citations_text, confidence, chunks


print("✅ RAG pipeline functions defined!")

# Quick test
print("\n🧪 Quick test retrieval:")
test_chunks = retrieve_chunks("What is the first-line treatment for hypertension?", top_k=3)
for i, c in enumerate(test_chunks, 1):
    print(f"  [{i}] {c['metadata']['title']} (relevance: {c['relevance_score']:.2%})")

## Cell 6: SHAP Explainability Function

In [ ]:
# Cell 6: SHAP Explainability Engine
# Computes document-level Shapley values showing each chunk's contribution

def get_embedding_score(query, response_text):
    """Score a response via cosine similarity between response and query embeddings."""
    if not response_text or not response_text.strip():
        return 0.0
    embeddings = embedding_model.encode([query, response_text])
    query_emb = embeddings[0].reshape(1, -1)
    response_emb = embeddings[1].reshape(1, -1)
    similarity = cosine_similarity(query_emb, response_emb)[0][0]
    return float(max(0.0, similarity))


def shap_scoring_function(masks, query, chunks):
    """Scoring function for SHAP KernelExplainer."""
    scores = []
    response_cache = {}

    for mask_row in masks:
        mask = mask_row.astype(bool).tolist()
        cache_key = str(mask)

        if cache_key in response_cache:
            response = response_cache[cache_key]
        else:
            active_chunks = [c for c, m in zip(chunks, mask) if m]
            if not active_chunks:
                response = ""
            else:
                try:
                    response = generate_answer(query, active_chunks)
                except Exception as e:
                    logger.warning(f"Generation failed for SHAP mask: {e}")
                    response = ""
            response_cache[cache_key] = response

        score = get_embedding_score(query, response)
        scores.append(score)

    return np.array(scores)


def make_chunk_label(chunk):
    """Create a short label for a document chunk."""
    title = chunk["metadata"].get("title", "Unknown")
    if len(title) > 35:
        title = title[:32] + "..."
    category = chunk["metadata"].get("category", "")
    return f"{title} [{category}]"


def compute_shap_values(query, chunks, nsamples=32):
    """
    Compute SHAP values for document-level attribution.
    Returns (labels, shap_values, base_value, explanation_text).
    """
    if not chunks:
        return [], [], 0.0, "No documents to analyze."

    labels = [make_chunk_label(c) for c in chunks]

    if len(chunks) < 2:
        return (
            labels, [1.0], 0.0,
            "Only one document retrieved — it accounts for 100% of the answer.",
        )

    n_features = len(chunks)
    background = np.zeros((1, n_features))
    instance = np.ones((1, n_features))

    def score_fn(masks):
        return shap_scoring_function(masks, query, chunks)

    try:
        explainer = shap.KernelExplainer(score_fn, background)
        actual_nsamples = min(nsamples, 2 ** n_features)
        shap_values = explainer.shap_values(
            instance, nsamples=actual_nsamples, silent=True,
        )

        values = (
            shap_values[0].tolist()
            if hasattr(shap_values[0], "tolist")
            else list(shap_values[0])
        )
        base_value = float(explainer.expected_value)

        # Build explanation text
        sorted_pairs = sorted(
            zip(labels, values, chunks),
            key=lambda x: abs(x[1]),
            reverse=True,
        )
        lines = ["**Document Contribution Analysis (SHAP)**\n"]
        lines.append(
            f"Base answer quality score (without any documents): {base_value:.3f}\n"
        )
        for label, value, chunk in sorted_pairs:
            direction = "↑ improves" if value > 0 else "↓ reduces"
            abs_val = abs(value)
            strength = (
                "strongly" if abs_val > 0.1
                else "moderately" if abs_val > 0.05
                else "slightly"
            )
            ev = chunk["metadata"].get("evidence_level", "N/A")
            lines.append(
                f"• **{label}** ({ev}): "
                f"{direction} answer quality by {abs_val:.3f} ({strength})"
            )
        if sorted_pairs:
            top_label, top_value, top_chunk = sorted_pairs[0]
            lines.append(
                f"\n**Key Insight:** The most influential source is "
                f"*{top_chunk['metadata'].get('title', 'Unknown')}*, "
                f"which {'positively' if top_value > 0 else 'negatively'} "
                f"contributes {abs(top_value):.3f} to the answer quality."
            )

        explanation_text = "\n".join(lines)
        return labels, [round(v, 4) for v in values], round(base_value, 4), explanation_text

    except Exception as e:
        logger.error(f"SHAP computation failed: {e}")
        # Fallback: use relevance scores as proxy
        relevances = [c["relevance_score"] for c in chunks]
        total = sum(relevances) if sum(relevances) > 0 else 1.0
        normalized = [round(r / total, 4) for r in relevances]
        explanation_text = (
            "**Document Contribution Analysis (Relevance-based Fallback)**\n\n"
            "SHAP computation was not available. Contributions estimated from retrieval relevance.\n\n"
            + "\n".join(
                f"• **{label}**: contribution weight {value:.3f}"
                for label, value in zip(labels, normalized)
            )
        )
        return labels, normalized, 0.0, explanation_text


def create_shap_bar_chart(labels, shap_values, base_value):
    """Create a horizontal bar chart of SHAP values and return as PIL Image."""
    if not labels or not shap_values:
        # Return a blank figure with a message
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.text(0.5, 0.5, "No SHAP data available",
                ha="center", va="center", fontsize=14, color="#888")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis("off")
        fig.patch.set_facecolor("#0f1419")
        buf = BytesIO()
        fig.savefig(buf, format="png", bbox_inches="tight", dpi=120,
                    facecolor=fig.get_facecolor())
        plt.close(fig)
        buf.seek(0)
        from PIL import Image
        return Image.open(buf)

    # Sort by SHAP value
    sorted_data = sorted(zip(labels, shap_values), key=lambda x: x[1])
    sorted_labels, sorted_values = zip(*sorted_data)

    # Color: positive = teal/green, negative = coral/red
    colors = ["#00d4aa" if v >= 0 else "#ff6b6b" for v in sorted_values]

    fig, ax = plt.subplots(figsize=(10, max(3, len(labels) * 0.7 + 1)))
    fig.patch.set_facecolor("#0f1419")
    ax.set_facecolor("#0f1419")

    bars = ax.barh(range(len(sorted_labels)), sorted_values, color=colors,
                   edgecolor="none", height=0.6, alpha=0.9)

    # Add value labels on bars
    for bar, val in zip(bars, sorted_values):
        x_pos = bar.get_width()
        ha = "left" if val >= 0 else "right"
        offset = 0.003 if val >= 0 else -0.003
        ax.text(x_pos + offset, bar.get_y() + bar.get_height() / 2,
                f"{val:+.4f}", va="center", ha=ha,
                fontsize=9, color="#e8e8e8", fontweight="bold")

    ax.set_yticks(range(len(sorted_labels)))
    ax.set_yticklabels(sorted_labels, fontsize=9, color="#e8e8e8")
    ax.set_xlabel("SHAP Value (impact on answer quality)",
                  fontsize=11, color="#e8e8e8", labelpad=10)
    ax.set_title(f"Document Contribution Analysis  |  Base value: {base_value:.4f}",
                 fontsize=13, color="#ffffff", fontweight="bold", pad=15)

    # Add zero line
    ax.axvline(x=0, color="#555555", linewidth=0.8, linestyle="-")

    # Style axes
    ax.tick_params(colors="#aaaaaa", which="both")
    for spine in ax.spines.values():
        spine.set_color("#333333")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="#00d4aa", label="Positive contribution"),
        Patch(facecolor="#ff6b6b", label="Negative contribution"),
    ]
    ax.legend(handles=legend_elements, loc="lower right",
              fontsize=9, facecolor="#1a2028", edgecolor="#333333",
              labelcolor="#e8e8e8")

    plt.tight_layout()

    # Convert to PIL Image
    buf = BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120,
                facecolor=fig.get_facecolor())
    plt.close(fig)
    buf.seek(0)
    from PIL import Image
    return Image.open(buf)


print("✅ SHAP explainability functions defined!")

## Cell 7: Gradio UI

Launch the interactive Healthcare Knowledge Navigator interface.

In [ ]:
# Cell 7: Gradio UI — Healthcare Knowledge Navigator

def process_query(query, top_k, enable_shap):
    """
    Main handler: runs RAG pipeline + optional SHAP analysis.
    Returns (answer, citations, confidence_text, shap_image, shap_explanation).
    """
    if not query or len(query.strip()) < 3:
        return (
            "⚠️ Please enter a medical question (at least 3 characters).",
            "", "", None, "",
        )

    # Run RAG pipeline
    answer, citations_text, confidence, chunks = rag_query(query, top_k=int(top_k))

    # Format confidence
    if confidence:
        conf_lines = [
            f"### Overall Confidence: {confidence.get('overall', 0):.1%}\n",
            f"| Metric | Score |",
            f"|--------|-------|",
            f"| 🎯 Retrieval Relevance | {confidence.get('retrieval_relevance', 0):.1%} |",
            f"| 🤝 Source Agreement | {confidence.get('source_agreement', 0):.1%} |",
            f"| 📊 Evidence Quality | {confidence.get('evidence_quality', 0):.1%} |",
            f"| 📋 Coverage | {confidence.get('coverage', 0):.1%} |",
        ]
        confidence_text = "\n".join(conf_lines)
    else:
        confidence_text = "No confidence data available."

    # SHAP analysis
    shap_image = None
    shap_explanation = ""

    if enable_shap and chunks:
        labels, shap_vals, base_val, explanation = compute_shap_values(
            query, chunks, nsamples=32,
        )
        if labels and shap_vals:
            shap_image = create_shap_bar_chart(labels, shap_vals, base_val)
            shap_explanation = explanation
        else:
            shap_explanation = "SHAP analysis could not be completed."

    return answer, citations_text, confidence_text, shap_image, shap_explanation


# --- Build the Gradio Interface ---

EXAMPLE_QUERIES = [
    ["What is the first-line treatment for type 2 diabetes?"],
    ["How should acute STEMI be managed in the emergency department?"],
    ["What are the four pillars of heart failure treatment?"],
    ["What antibiotics should be used for community-acquired pneumonia?"],
    ["What are the blood pressure targets for patients with CKD?"],
    ["How is septic shock managed per the Surviving Sepsis Campaign?"],
    ["What is the role of SGLT2 inhibitors in CKD?"],
    ["Compare COPD and asthma management approaches"],
]

with gr.Blocks(
    title="Healthcare Knowledge Navigator",
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.teal,
        secondary_hue=gr.themes.colors.cyan,
        neutral_hue=gr.themes.colors.slate,
        font=gr.themes.GoogleFont("Inter"),
    ),
    css="""
        .gradio-container { max-width: 1200px !important; }
        .main-header { text-align: center; padding: 20px 0; }
        .main-header h1 { font-size: 2.2em; margin-bottom: 0.2em; }
        .main-header p { font-size: 1.1em; opacity: 0.8; }
    """,
) as demo:

    # Header
    gr.HTML("""
        <div class="main-header">
            <h1>🏥 Healthcare Knowledge Navigator</h1>
            <p>Explainable RAG System for Evidence-Based Medical Knowledge</p>
            <p style="font-size: 0.85em; opacity: 0.6;">
                Powered by Gemini Flash · ChromaDB · SHAP Explainability · 15 Clinical Documents
            </p>
        </div>
    """)

    with gr.Row():
        with gr.Column(scale=3):
            query_input = gr.Textbox(
                label="🔍 Medical Question",
                placeholder="e.g., What is the first-line treatment for type 2 diabetes?",
                lines=2,
                max_lines=4,
            )
        with gr.Column(scale=1):
            top_k_slider = gr.Slider(
                minimum=3, maximum=10, value=5, step=1,
                label="📚 Documents to Retrieve",
            )
            enable_shap_checkbox = gr.Checkbox(
                value=True,
                label="🔬 Enable SHAP Analysis",
                info="Shows which documents contribute most (takes ~30-60s)",
            )
            submit_btn = gr.Button(
                "🚀 Analyze", variant="primary", size="lg",
            )

    # Example queries
    gr.Examples(
        examples=EXAMPLE_QUERIES,
        inputs=[query_input],
        label="💡 Example Queries",
    )

    # Results
    with gr.Tabs():
        with gr.Tab("📝 Answer"):
            answer_output = gr.Markdown(
                label="Answer",
                value="*Enter a question above and click Analyze to get started.*",
            )

        with gr.Tab("📚 Citations"):
            citations_output = gr.Markdown(
                label="Source Citations",
                value="*Citations will appear here after querying.*",
            )

        with gr.Tab("📊 Confidence"):
            confidence_output = gr.Markdown(
                label="Confidence Breakdown",
                value="*Confidence scores will appear here after querying.*",
            )

        with gr.Tab("🔬 SHAP Explainability"):
            shap_image_output = gr.Image(
                label="Document Contribution Chart",
                type="pil",
            )
            shap_text_output = gr.Markdown(
                label="SHAP Explanation",
                value="*Enable SHAP analysis and submit a query to see document contributions.*",
            )

    # Footer
    gr.HTML("""
        <div style="text-align: center; padding: 15px 0; opacity: 0.6; font-size: 0.85em;">
            ⚠️ For educational and clinical decision support purposes only.
            Not a substitute for professional medical judgment.
        </div>
    """)

    # Wire up the submit
    submit_btn.click(
        fn=process_query,
        inputs=[query_input, top_k_slider, enable_shap_checkbox],
        outputs=[answer_output, citations_output, confidence_output,
                 shap_image_output, shap_text_output],
    )
    query_input.submit(
        fn=process_query,
        inputs=[query_input, top_k_slider, enable_shap_checkbox],
        outputs=[answer_output, citations_output, confidence_output,
                 shap_image_output, shap_text_output],
    )

# Launch
demo.launch(share=True, debug=False)